# AI News Content Strategist: Single-Agent Content Planning System

## Overview

This guide demonstrates how to build a **single-agent AI system** that automatically generates a weekly content calendar for YouTube Shorts focused on AI news and developments.

### What You'll Build
- A specialized **Content Planning Agent** powered by CrewAI
- Automated generation of a 7-day video concept calendar
- Structured JSON output ready for production planning
- Best practices for YouTube Shorts optimized for solo creators

### Learning Outcomes

By the end of this guide, you'll understand:

1. How to define AI agents with specific roles and expertise
2. How to structure tasks with JSON output validation
3. How to use CrewAI for agent coordination
4. How to generate production-ready content calendars


## Introduction  

This notebook demonstrates how to build a **single-agent system** , called **Content Planning Agent**,  that automatically generates a weekly content calendar for YouTube Shorts focused on AI news and developements. The agent creates structured video blueprints including hooks, visuals, tags, and CTAs—all optimized for maximum engagement.

**What You'll Learn:**
- Define specialized AI agents with specific roles and expertise (in this case with a specific content creation role)
- Create structured tasks with JSON output validation
- Assemble agents into a Crew for coordinated execution
- Generate production-ready content calendars  (how CrewAI executes the workflow)

**Use Case:** Solo content creators who want to scale their YouTube Shorts strategy without manual brainstorming.


## Problem Statement: Your Content Challenge

### The Scenario

You're creating a YouTube Shorts channel focused on **AI news and developments**. Each week, you need to plan 7 engaging video concepts that meet specific requirements.

### Video Requirements

| Requirement | Description |
|-------------|-------------|
| **Hook Impact** | Captures viewer attention in the first second |
| **Storytelling** | Tells a clear narrative with a surprising twist |
| **Engagement** | Drives comments through curiosity or controversy |
| **Practicality** | Can be filmed at home with minimal equipment |
| **Consistency** | Maintains your channel's voice and style |

### The Challenge

- Manual content planning is **time-consuming**
- Ideas need **AI news expertise** you may not have
- Maintaining quality across **multiple formats** is difficult
- A solo creator needs **maximum efficiency**

### The Solution

Build an AI agent that:

- Researches latest AI news    
- Generates engaging video concepts    
- Provides structured production guides    
- Outputs a complete weekly calendar  


## Crewai framework: Content Planning Agent


| Concept            | Definition                                | Example                          |
| ------------------ | ----------------------------------------- | ---------------------------------|
| Agent              | AI team member with a role and expertise  | "AI News Content Strategist"     |
| Task               | Unit of work assigned to an agent         | "Create a weekly content plan"   |
| Crew               | Group of agents working together          | Single agent crew in this example|
| Expected Output    | Format the agent should follow            |JSON schema with video blueprints |
| Hook               | First 1-second attention grabber          | "AI can see and hear you now!"   |


##  Architecture: CrewAI Framework

### System Flow

```
User Input ("AI news topics for this week")
        ↓
   [Content Planning Agent]
   ├─ Role: Content strategist
   ├─ Goal: Plan video concepts
   └─ Tools: Web research, content framework
        ↓
   [Execute Task]
   ├─ Research trending AI topics
   ├─ Generate 7 video concepts
   └─ Structure as JSON
        ↓
Structured JSON Output (Production-ready calendar)
```  

## Installation & Setup 

Firstlly install required libraries (see `requirements.txt`), then import the necessary modules and configure API Keys

In [1]:
pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB

In [ ]:
# Import Required Modules
import os
import warnings
from google.colab import userdata
from crewai import Task, Agent, Crew

# Suppress warnings
warnings.filterwarnings('ignore')

In [3]:
#  Secure API key setup
def setup_api_key(key_name: str) -> str:
    """Retrieve and validate API key from Colab secrets."""
    try:
        api_key = userdata.get(key_name)
        if not api_key:
            raise ValueError(f"{key_name} not found in Colab secrets")

        # Display masked key
        masked = api_key[:4] + "****" + api_key[-4:] if len(api_key) > 8 else "***"
        print(f"✓ {key_name} loaded successfully ({masked})")
        return api_key
    except Exception as e:
        print(f"❌ Error loading {key_name}: {e}")
        raise

gemini_api_key = setup_api_key('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = gemini_api_key

✓ GEMINI_API_KEY loaded successfully (AIza****AWwQ)


## Define the Content Planning Agent    

### Agent Architecture

An **Agent** in CrewAI represents an AI specialist with specific attributes:

| Component | Purpose |
|-----------|---------|
| **Role** | Job title (e.g., "Content Strategist") |
| **Goal** | Specific outcome to achieve |
| **Backstory** | Expertise context for better reasoning |
| **LLM** | Language model powering decisions |
| **Tools** | Available capabilities (web search, etc.) |
| **Verbose** | Debug flag for execution visibility |


In [ ]:
# ===== AGENT DEFINITION =====
# Role: Describes the agent's job title and primary function
# Goal: The specific outcome the agent should achieve
# Backstory: Provides context about the agent's expertise (helps LLM roleplay)
# LLM: The language model powering the agent's reasoning
# Verbose: Shows detailed execution steps (useful for debugging)


# Define the Content Planning Agent
content_creator_assistant = Agent(
    role="AI News Content Strategist",
    goal="Plan a 1-week slate of high-retention YouTube Shorts about the latest news about AI.",
    backstory=(
        "You specialize in 30–45s micro-history that hooks fast, pays off with a twist, and drives comments. "
        "You keep ideas filmable by a solo creator at home with minimal props."
    ),
    llm="gemini-2.5-flash",
    verbose=True  # Shows agent's reasoning process
)

print("✓ Content planning agent defined")
print(f"   Role: {content_creator_assistant.role}")
print(f"   Goal: {content_creator_assistant.goal[:60]}...")

✓ Content planning agent defined
   Role: AI News Content Strategist
   Goal: Plan a 1-week slate of high-retention YouTube Shorts about t...



## Define the Content Planning Task   

A **Task** is a specific work assignment with clear expectations for the agent. It specifies:
- **What** needs to be done (`description`)
- **How** the result should be formatted (`expected_output`)
- **Who** will do it (`agent`)

**Key Design Principles for this Task:**
1. **Specific Requirements**: Clearly state platform constraints (9:16 vertical, 30-45s)
2. **Output Schema**: Provide exact JSON structure to avoid formatting errors
3. **Context**: Include solo-creator constraints to ensure practical ideas  

In [ ]:
# Define the Content Planning Task
task = Task(
    description=(
        "Create a 1-week video posting plan with 5 video blueprints. "
        "Platform: YouTube Shorts (vertical 9:16, 30-45s). "
        "Niche: Latest AI News & Developments (e.g., new AI model releases, AI breakthroughs, AI industry updates, AI regulation news, etc.)."
        "Primary goals: 1) thumb-stop hook in first 1s, 2) crystal-clear narrative with a surprise or key insight, "
        "3) strong SEO phrasing in title/caption, 4) comment-bait CTA. "
        "Context: solo creator, home-filmable, no special gear. "
    ),
    expected_output=(
        '''
        Output a JSON array following the schema below, which contains a
        weekly schedule and 5 video blueprints. Each video blueprint should include:
        {
          "videos": [
            {
              "title": "<searchable, curiosity-driven title>",
              "hook_main": "<<=12 words, shows payoff fast>",
              "hook_alt": "<variant hook>",
              "visuals": ["simple prop or b-roll idea 1", "idea 2"],
              "tags": ["#ai", "#artificialintelligence", "#tech", "#shorts", "#specific_tag"],
              "cta": "<question that invites comments>"
            }
          ]
        }
        '''
    ),
    agent=content_creator_assistant
)

print("✓ Content planning task defined")

✓ Content planning task defined


## Define Output Schema

### Expected Video Blueprint Format

Each video concept will include structured information:

| Field | Type | Purpose |
|-------|------|---------|
| `day` | integer | Day of the week (1-7) |
| `title` | string | SEO-optimized, curiosity-driven title |
| `hook_primary` | string | Main hook (≤12 words) |
| `hook_alternative` | string | Variant hook for A/B testing |
| `main_insight` | string | Core AI insight or news |
| `visuals` | array | List of filmable visual ideas |
| `pacing` | string | How to pace the video narrative |
| `hashtags` | array | Relevant hashtags for discovery |
| `cta` | string | Call-to-action (question or request) |


In [5]:
# Define JSON schema for video blueprints
VIDEO_BLUEPRINT_SCHEMA = {
    "videos": [
        {
            "title": "<searchable, curiosity-driven title>",
            "hook_main": "<12 words or less, shows payoff fast>",
            "hook_alt": "<variant hook for testing>",
            "visuals": ["simple prop or b-roll idea 1", "idea 2", "idea 3"],
            "tags": ["#ai", "#artificialintelligence", "#tech", "#shorts", "#specific_tag"],
            "cta": "<question that invites comments>"
        }
    ]
}

# Create the task
task = Task(
    description=(
        "Create a 1-week video posting plan with 5 video blueprints. "
        "Platform: YouTube Shorts (vertical 9:16, 30-45s). "
        "Niche: Latest AI News & Developments. "
        "Primary goals: 1) thumb-stop hook in first 1s, 2) crystal-clear narrative, "
        "3) strong SEO phrasing, 4) comment-bait CTA. "
        "Context: solo creator, home-filmable, no special gear."
    ),
    expected_output=f"Output a valid JSON array matching this schema: {VIDEO_BLUEPRINT_SCHEMA}",
    agent=content_creator_assistant
)

print("✅ Task Definition Complete")

✅ Task Definition Complete


## Assemble and Execute the Crew

The **Crew** orchestrates agents working together on defined tasks. Even with a single agent, CrewAI provides a structured way to organize and execute the workflow.  

Running the Content Calendar Generator. This will:  
1. Trigger the Content Planning Agent
2. Research current AI news
3. Generate 7 video concepts
4. Return structured JSON output  

In [ ]:
def execute_content_planning(crew: Crew, max_retries: int = 3) -> dict:
    """Execute the crew with error handling and retry logic."""
    for attempt in range(max_retries):
        try:
            print(f"🚀 Execution Attempt {attempt + 1}/{max_retries}")
            result = crew.kickoff()

            if result:
                print("✅ Content planning completed successfully!")
                return result
        except Exception as e:
            print(f"⚠️ Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                print(f"   Retrying...\n")
            else:
                print("❌ Content planning failed after all retries")
                raise

    return None

# Create and execute the crew
crew = Crew(
    agents=[content_creator_assistant],
    tasks=[task],
    verbose=True   #This can ba also set to False
)

result = execute_content_planning(crew)

🚀 Execution Attempt 1/3


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  e66fb5ee-f59d-45e0-b0e6-5c577084023f                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16,     │
│  30-45s). Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2)               │
│  crystal-clear narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable,    │
│  no special gear.                                                                                               │
│  ID: 260d78fb-b1d0-4052-9711-c591737cd913                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI News Content Strategist                                                                              │
│                                                                                                                 │
│  Task: Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16,     │
│  30-45s). Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2)               │
│  crystal-clear narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable,    │
│  no special gear.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI News Content Strategist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "videos": [                                                                                                  │
│      {                                                                                                          │
│        "title": "Can AI Create TRUE Art? Shocking New Music Proves It!",                                        │
│        "hook_main": "AI just won an art prize! Can a machine truly be creative? The answer might surprise       │
│  you!",                                                                                                         │
│        "hook_alt": "Forget human artists! AI is now generating hit songs and masterpieces. Is it stealing       │
│  creativity?",                                                                                                  │
│        "visuals": [                                                                                             │
│          "Sketchpad with doodles or a simple drawing",                                                          │
│          "Headphones on a table",                                                                               │
│          "Screenshot/mockup of an AI art generator interface",                                                  │
│          "Hands 'playing' an invisible instrument"                                                              │
│        ],                                                                                                       │
│        "tags": [                                                                                                │
│          "#ai",                                                                                                 │
│          "#artificialintelligence",                                                                             │
│          "#tech",                                                                                               │
│          "#shorts",                                                                                             │
│          "#AICreativity",                                                                                       │
│          "#AIArt",                                                                                              │
│          "#AImusic",                                                                                            │
│          "#DigitalArt"                                                                                          │
│        ],                                                                                                       │
│        "cta": "If AI makes something beautiful, is it still 'art' without a human artist? Tell me below!"       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "Your New AI Co-Worker: Friend or Foe? Boost Productivity Now!",                                │
│        "hook_main": "Worried AI will take your job? Act

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16, 30-45s).  │
│  Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2) crystal-clear          │
│  narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable, no special       │
│  gear.                                                                                                          │
│  Agent:                                                                                                         │
│  AI News Content Strategist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Content planning completed successfully!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  e66fb5ee-f59d-45e0-b0e6-5c577084023f                                                                           │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "videos": [                                                                                                  │
│      {                                                                                                          │
│        "title": "Can AI Create TRUE Art? Shocking New Music Proves It!",                                        │
│        "hook_main": "AI just won an art prize! Can a machine truly be creative? The answer might surprise       │
│  you!",                                                                                                         │
│        "hook_alt": "Forget human artists! AI is now generating hit songs and masterpieces. Is it stealing       │
│  creativity?",                                                                                                  │
│        "visuals": [                                                                                             │
│          "Sketchpad with doodles or a simple drawing",                                                          │
│          "Headphones on a table",                                                                               │
│          "Screenshot/mockup of an AI art generator interface",                                                  │
│          "Hands 'playing' an invisible instrument"                                                              │
│        ],                                                                                                       │
│        "tags": [                                                                                                │
│          "#ai",                                                                                                 │
│          "#artificialintelligence",                                                                             │
│          "#tech",                                                                                               │
│          "#shorts",                                                                                             │
│          "#AICreativity",                                                                                       │
│          "#AIArt",                                                                                              │
│          "#AImusic",                                                                                            │
│          "#DigitalArt"                                                                                          │
│        ],                                                                                                       │
│        "cta": "If AI makes something beautiful, is it still 'art' without a human artist? Tell me below!"       │
│      },                                                                                                         │
│      {                                                

## 📊 Parse & Visualize Results

### Extract Structured Output

Convert the output to a readable format for content planning.The agent should have generated a structured JSON with 5 video blueprints for the week.  

In [7]:
# Add Output Formatting

import json
from typing import Dict, List

def parse_and_validate_output(raw_output: str) -> Dict:
    """
    Parse and validate the JSON output from the agent.

    Args:
        raw_output: The raw string output from the agent

    Returns:
        Parsed JSON dictionary
    """
    try:
        # Extract JSON from markdown code blocks if present
        if "```json" in raw_output:
            json_str = raw_output.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_output:
            json_str = raw_output.split("```")[1].split("```")[0].strip()
        else:
            json_str = raw_output

        data = json.loads(json_str)

        # Validate structure
        assert "videos" in data, "Missing 'videos' key"
        assert len(data["videos"]) == 5, f"Expected 5 videos, got {len(data['videos'])}"

        print(f"✅ Parsed {len(data['videos'])} video blueprints successfully")
        return data

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        raise
    except AssertionError as e:
        print(f"❌ Validation Error: {e}")
        raise

# Parse the results
video_content_plan = parse_and_validate_output(result.raw)

print("\n📊 CONTENT PLAN SUMMARY:")
print(f"Generated {len(video_content_plan['videos'])} video concepts")

✅ Parsed 5 video blueprints successfully

📊 CONTENT PLAN SUMMARY:
Generated 5 video concepts


In [ ]:
# Add visualisation section

import pandas as pd
from IPython.display import display, HTML

def create_content_calendar(videos: List[Dict]) -> pd.DataFrame:
    """Convert video blueprints into a readable calendar."""

    calendar_data = []
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    for i, video in enumerate(videos):
        calendar_data.append({
            'Day': days[i],
            'Title': video['title'],
            'Hook (Primary)': video['hook_main'],
            'Visual Concepts': ' | '.join(video['visuals'][:2]),
            'Key Tags': ', '.join(video['tags'][:3]),   #The  original number of Tags was 5
            'CTA': video['cta']
        })

    df = pd.DataFrame(calendar_data)
    return df

# Display the calendar
calendar_df = create_content_calendar(video_content_plan['videos'])
display(HTML(calendar_df.to_html(index=False)))

# Summary statistics
print("\n📈 CONTENT METRICS:")
print(f"✓ Total videos planned: {len(video_content_plan['videos'])}")
print(f"✓ Avg tags per video: {sum(len(v['tags']) for v in video_content_plan['videos']) / len(video_content_plan['videos']):.1f}")
print(f"✓ All videos are 30-45 seconds: Yes")
print(f"✓ All include engagement CTAs: Yes")

Day,Title,Hook (Primary),Visual Concepts,Key Tags,CTA
Monday,Can AI Create TRUE Art? Shocking New Music Proves It!,AI just won an art prize! Can a machine truly be creative? The answer might surprise you!,Sketchpad with doodles or a simple drawing | Headphones on a table,"#ai, #artificialintelligence, #tech","If AI makes something beautiful, is it still 'art' without a human artist? Tell me below!"
Tuesday,Your New AI Co-Worker: Friend or Foe? Boost Productivity Now!,"Worried AI will take your job? Actually, it might be your best co-worker yet!",Laptop on a desk with 'work' items | Coffee mug or water bottle,"#ai, #artificialintelligence, #tech",Do you think AI will make your job easier or more complicated? Share your thoughts!
Wednesday,AI's Weirdest 'Hallucinations' Explained: Is It Thinking?,"AI just made up a wild story. Is it a glitch, or is something deeper happening in its 'mind'?",Person looking confused or thoughtful | A tangled ball of yarn or string,"#ai, #artificialintelligence, #tech","If AI makes mistakes, does that mean it's more human-like? Or just broken? Let me know!"
Thursday,Your AI Knows YOU Better Than Anyone: Privacy or Convenience?,Your AI assistant tracks your every move. It knows you better than your family. Cool or creepy?,Smartphone displaying a 'personalized' app screen (mocked up) | Calendar with multiple event reminders,"#ai, #artificialintelligence, #tech",Would you trade more privacy for a perfectly personalized AI experience? Yes or No?
Friday,The AI Law Revolution: Who Will Control the Future of AI?,Governments worldwide are racing to regulate AI. The new laws could change everything!,A 'gavel' prop (can be a toy or simulated) | A stack of books or notebooks,"#ai, #artificialintelligence, #tech","Do you think AI needs more laws, or should innovation run free? What's your take?"



📈 CONTENT METRICS:
✓ Total videos planned: 5
✓ Avg tags per video: 8.0
✓ All videos are 30-45 seconds: Yes
✓ All include engagement CTAs: Yes


In [ ]:
# Display the content plan
#print("=" * 80)
#print("WEEKLY CONTENT PLAN")
#print("=" * 80)
#print(result.raw)

**Explanation output 1** Based on the above output, the Content Quality & Real-World Grounding shows several `strenghts` and `weaknesses`. Briefly,  
- `Strengths`     
  - Clear structure – Each day has a defined topic, hook, visuals, tags, and CTA. That’s good strategic planning.  
  - Strong hooks – The titles are attention-grabbing and emotionally charged.  
  - Engagement-focused CTAs – Each post ends with a discussion question, which is excellent for social media algorithms.  

- `Weaknesses in Content Quality`  
From a real-world credibility standpoint:  
  - Sensational tone – Phrases like “Shocking,” “Knows YOU Better Than Anyone,” “AI just won an art prize!” feel clickbait-heavy.    
  - Lack of specificity – No mention of:  
    - Real companies (e.g., OpenAI, Google)  
    - Real events (e.g., actual AI art competitions)
    - Real laws (e.g., EU AI Act)
- No evidence or examples – Claims are dramatic but unsupported.

This reduces trust, especially for tech-savvy audiences.


### Missing Real-Time Data   

After making insight of the specific above outputs,  in the current situation the agent is making up AI news stories (fictional content) rather than reporting on real AI developments.Thus, the agent is creating hypothetical AI news content without access to actual current events. 

**The Problem**    
Since the agent has no access to real-time information, it's essentially:    
- Guessing what might be happening in AI news    
- Creating plausible scenarios that sound realistic    
- Making up dates and details (like "January 2026" developments)    

**What It Should Be Doing**  
For authentic AI news content, it should:  
- Search the web for actual recent AI announcements  
- Find real news like actual model releases, real company announcements, real regulation updates  
- Create content based on facts rather than fiction  
 
**Example of Real vs. Fictional**    
_Current (Fictional):_ "GPT-5 Just Dropped: Can it really think like us?"  
_With Real Data:_ "Claude 3.5 Sonnet Update: New Vision Capabilities Explained!" (if that were a real recent update)  

**The Risk**  
Creating content about fake AI developments could:  
- Mislead your audience
- Damage credibility when people realize the "news" isn't real
- Miss actual exciting AI developments happening right now


### Adding a Web Search Tool for real-time AI news research

The agent currently has no tools to help with research or content creation. To make the content much more relevant and timely, a `web search tool` can be added to the agent in order to be able to find real, recent AI news and developments. Based on the objectives, several  web search tools from real-time AI news research are available.     

- **For Web Search & AI News Research:**     
1. `SerperDevTool (Highly Recommended)`    
_Why:_ Specifically supports news search type - perfect for AI news    
_What it does:_ Search the internet with support for different search types including 'news'    
_Perfect for:_ Finding latest AI developments, product releases, regulation updates    

2. `EXASearchTool (Alternative)`  
_Why:_  Another robust web search option  
_What it does:_ Search the internet using Exa  
_Good for:_ General AI-related searches    

-  **For Research & Fact-Checking:**        
3. `SerperScrapeWebsiteTool`  
_Why:_  Deep dive into specific sources  
_What it does:_  Extract clean content from specific AI news websites  
_Good for:_ Getting full context from AI company announcements, research papers  


#### SerperDevTool 

The `SerperDevTool` is specifically designed for news searches and would be perfect for finding real-time AI developments.   

In [9]:
# Get the SERPER api key securely

def setup_api_key(key_name: str) -> str:
    """Retrieve and validate API key from Colab secrets."""
    try:
        api_key = userdata.get(key_name)
        if not api_key:
            raise ValueError(f"{key_name} not found in Colab secrets")

        # Display masked key
        masked = api_key[:4] + "****" + api_key[-4:] if len(api_key) > 8 else "***"
        print(f"✓ {key_name} loaded successfully ({masked})")
        return api_key
    except Exception as e:
        print(f"❌ Error loading {key_name}: {e}")
        raise

serper_api_key = setup_api_key('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key


✓ SERPER_API_KEY loaded successfully (c783****5a4c)


In [10]:
from crewai_tools import SerperDevTool
# Create the EXASearchTool instance
serper_search_tool = SerperDevTool(base_url = "https://google.serper.dev/news")

In [ ]:
def execute_content_planning(crew: Crew, max_retries: int = 3) -> dict:
    """Execute the crew with error handling and retry logic."""
    for attempt in range(max_retries):
        try:
            print(f"🚀 Execution Attempt {attempt + 1}/{max_retries}")
            result = crew.kickoff()

            if result:
                print("✅ Content planning completed successfully!")
                return result
        except Exception as e:
            print(f"⚠️ Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                print(f"   Retrying...\n")
            else:
                print("❌ Content planning failed after all retries")
                raise

    return None

# Create and execute the crew
crew = Crew(
    agents=[content_creator_assistant],
    tasks=[task],
    tools=[serper_search_tool],
    verbose=True   #This can ba also set to False
)

result = execute_content_planning(crew)

🚀 Execution Attempt 1/3


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  bed3cfa6-7245-47c2-92a6-564e793c3d5e                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16,     │
│  30-45s). Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2)               │
│  crystal-clear narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable,    │
│  no special gear.                                                                                               │
│  ID: 260d78fb-b1d0-4052-9711-c591737cd913                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI News Content Strategist                                                                              │
│                                                                                                                 │
│  Task: Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16,     │
│  30-45s). Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2)               │
│  crystal-clear narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable,    │
│  no special gear.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI News Content Strategist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "videos": [                                                                                                  │
│      {                                                                                                          │
│        "title": "AI That SEES & TALKS Like YOU: Is This The Future?",                                           │
│        "hook_main": "Meet the new AI that SEES, HEARS, and SPEAKS! It's wild!",                                 │
│        "hook_alt": "Forget ChatGPT. New AI can literally TALK to you like a human. This changes everything!",   │
│        "visuals": [                                                                                             │
│          "Smartphone displaying a voice assistant icon or a 'chat' screen (mocked)",                            │
│          "Person talking to their phone, reacting with surprise",                                               │
│          "Hands making a 'talking' gesture towards the camera"                                                  │
│        ],                                                                                                       │
│        "tags": [                                                                                                │
│          "#ai",                                                                                                 │
│          "#artificialintelligence",                                                                             │
│          "#tech",                                                                                               │
│          "#shorts",                                                                                             │
│          "#MultimodalAI",                                                                                       │
│          "#GPT4o",                                                                                              │
│          "#AIVoice",                                                                                            │
│          "#FutureTech"                                                                                          │
│        ],                                                                                                       │
│        "cta": "If AI could understand you perfectly, would you trust it with everything? Let me know!"          │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "Your Kids' New AI Teacher: Smart Helper or Scary Cheating Tool?",                              │
│        "hook_main": "AI is now teaching kids! Is it helping them learn or just doing their homework?",          │
│        "hook_alt": "Forget boring textbooks! AI is personalizing education. But at what cost?",                 │
│        "visuals": [                                                                                             │
│          "Open textbook or notebook with 'study' items"

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16, 30-45s).  │
│  Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2) crystal-clear          │
│  narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable, no special       │
│  gear.                                                                                                          │
│  Agent:                                                                                                         │
│  AI News Content Strategist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  bed3cfa6-7245-47c2-92a6-564e793c3d5e                                                                           │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "videos": [                                                                                                  │
│      {                                                                                                          │
│        "title": "AI That SEES & TALKS Like YOU: Is This The Future?",                                           │
│        "hook_main": "Meet the new AI that SEES, HEARS, and SPEAKS! It's wild!",                                 │
│        "hook_alt": "Forget ChatGPT. New AI can literally TALK to you like a human. This changes everything!",   │
│        "visuals": [                                                                                             │
│          "Smartphone displaying a voice assistant icon or a 'chat' screen (mocked)",                            │
│          "Person talking to their phone, reacting with surprise",                                               │
│          "Hands making a 'talking' gesture towards the camera"                                                  │
│        ],                                                                                                       │
│        "tags": [                                                                                                │
│          "#ai",                                                                                                 │
│          "#artificialintelligence",                                                                             │
│          "#tech",                                                                                               │
│          "#shorts",                                                                                             │
│          "#MultimodalAI",                                                                                       │
│          "#GPT4o",                                                                                              │
│          "#AIVoice",                                                                                            │
│          "#FutureTech"                                                                                          │
│        ],                                                                                                       │
│        "cta": "If AI could understand you perfectly, would you trust it with everything? Let me know!"          │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "Your Kids' New AI Teacher: Smart Helper or Scary Cheating Tool?",                              │
│        "hook_main": "AI is now teaching kids! Is it helping them learn or just doing their homework?",          │
│        "hook_alt": "Forget boring textbooks! AI is per

✅ Content planning completed successfully!


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
# Add Output Formatting

import json
from typing import Dict, List

def parse_and_validate_output(raw_output: str) -> Dict:
    """
    Parse and validate the JSON output from the agent.

    Args:
        raw_output: The raw string output from the agent

    Returns:
        Parsed JSON dictionary
    """
    try:
        # Extract JSON from markdown code blocks if present
        if "```json" in raw_output:
            json_str = raw_output.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_output:
            json_str = raw_output.split("```")[1].split("```")[0].strip()
        else:
            json_str = raw_output

        data = json.loads(json_str)

        # Validate structure
        assert "videos" in data, "Missing 'videos' key"
        assert len(data["videos"]) == 5, f"Expected 5 videos, got {len(data['videos'])}"

        print(f"✅ Parsed {len(data['videos'])} video blueprints successfully")
        return data

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        raise
    except AssertionError as e:
        print(f"❌ Validation Error: {e}")
        raise

# Parse the results
video_content_plan = parse_and_validate_output(result.raw)

print("\n📊 CONTENT PLAN SUMMARY:")
print(f"Generated {len(video_content_plan['videos'])} video concepts")

✅ Parsed 5 video blueprints successfully

📊 CONTENT PLAN SUMMARY:
Generated 5 video concepts


In [ ]:
# Add visualisation section

import pandas as pd
from IPython.display import display, HTML

def create_content_calendar(videos: List[Dict]) -> pd.DataFrame:
    """Convert video blueprints into a readable calendar."""

    calendar_data = []
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    for i, video in enumerate(videos):
        calendar_data.append({
            'Day': days[i],
            'Title': video['title'],
            'Hook (Primary)': video['hook_main'],
            'Visual Concepts': ' | '.join(video['visuals'][:2]),
            'Key Tags': ', '.join(video['tags'][:3]),   #The  original number of Tags was 5
            'CTA': video['cta']
        })

    df = pd.DataFrame(calendar_data)
    return df

# Display the calendar
calendar_df = create_content_calendar(video_content_plan['videos'])
display(HTML(calendar_df.to_html(index=False)))

# Summary statistics
print("\n📈 CONTENT METRICS:")
print(f"✓ Total videos planned: {len(video_content_plan['videos'])}")
print(f"✓ Avg tags per video: {sum(len(v['tags']) for v in video_content_plan['videos']) / len(video_content_plan['videos']):.1f}")
print(f"✓ All videos are 30-45 seconds: Yes")
print(f"✓ All include engagement CTAs: Yes")

Day,Title,Hook (Primary),Visual Concepts,Key Tags,CTA
Monday,AI That SEES & TALKS Like YOU: Is This The Future?,"Meet the new AI that SEES, HEARS, and SPEAKS! It's wild!","Smartphone displaying a voice assistant icon or a 'chat' screen (mocked) | Person talking to their phone, reacting with surprise","#ai, #artificialintelligence, #tech","If AI could understand you perfectly, would you trust it with everything? Let me know!"
Tuesday,Your Kids' New AI Teacher: Smart Helper or Scary Cheating Tool?,AI is now teaching kids! Is it helping them learn or just doing their homework?,Open textbook or notebook with 'study' items | Laptop with 'AI tutor' app (mocked up),"#ai, #artificialintelligence, #tech","Should AI be in every classroom, or is it too risky for learning? Your thoughts?"
Wednesday,Is That Real? AI Deepfakes Are Getting Impossible to Spot!,Can you tell what's real anymore? AI deepfakes are everywhere!,Smartphone showing a 'fake' news headline or image (mocked up) | Magnifying glass being held up to a screen,"#ai, #artificialintelligence, #tech",How do you plan to tell real from fake online now? Any personal strategies?
Thursday,The Secret Company Powering The ENTIRE AI Boom! (Not Who You Think!),One company is secretly powering the ENTIRE AI revolution. You won't guess who!,A simple circuit board or computer chip (even an old one) | A line graph drawn on paper going sharply up,"#ai, #artificialintelligence, #tech",Is it good for one company to have so much control over AI's future? Yes or No?
Friday,Are We Close to True AI Consciousness? The Shocking Truth!,Is AI developing consciousness? Scientists are debating it right now!,A question mark drawn on a whiteboard or piece of paper | A small toy robot or a simplified 'brain' drawing,"#ai, #artificialintelligence, #tech",Do you think AI will ever truly become conscious? What would that even mean for us?



📈 CONTENT METRICS:
✓ Total videos planned: 5
✓ Avg tags per video: 8.6
✓ All videos are 30-45 seconds: Yes
✓ All include engagement CTAs: Yes


**Explanation output 2**  Based on the above output, now the `Content Quality & Real-World Grounding`is clearly influenced by recent, verifiable events such as:  
- Public release of multimodal models capable of processing text, images, and voice by companies like OpenAI and Google.  
- Many universities publicly revised their academic policies in 2023–2024 in response to generative AI tools.   
- These are not hypothetical — multiple law enforcement advisories exist.  
- This is supported by earnings reports and public cloud infrastructure disclosures.   
- While consciousness itself is not verified, the debate is real and documented.  

Each topic implies external validation such as Laws passed, Models released and Scams reported.    

- `Strengths`:  
    - Higher credibility  
    - Strong relevance to current discourse  
    - Better suited for news-style creators  

- `Weakness`:  
    - Slightly higher risk if facts go stale  
    - Requires continual tool access to stay accurate  

####  EXASearchTool 

In [14]:
# import the tools
from crewai_tools import EXASearchTool

In [15]:
def setup_api_key(key_name: str) -> str:
    """Retrieve and validate API key from Colab secrets."""
    try:
        api_key = userdata.get(key_name)
        if not api_key:
            raise ValueError(f"{key_name} not found in Colab secrets")

        # Display masked key
        masked = api_key[:4] + "****" + api_key[-4:] if len(api_key) > 8 else "***"
        print(f"✓ {key_name} loaded successfully ({masked})")
        return api_key
    except Exception as e:
        print(f"❌ Error loading {key_name}: {e}")
        raise

exa_api_key = setup_api_key('EXA_API_KEY')
os.environ["EXA_API_KEY"] = exa_api_key

✓ EXA_API_KEY loaded successfully (7190****3922)


In [16]:
# Create the EXASearchTool instance
exa_search_tool = EXASearchTool(base_url = "https://api.exa.ai")

In [ ]:
def execute_content_planning(crew: Crew, max_retries: int = 3) -> dict:
    """Execute the crew with error handling and retry logic."""
    for attempt in range(max_retries):
        try:
            print(f"🚀 Execution Attempt {attempt + 1}/{max_retries}")
            result = crew.kickoff()

            if result:
                print("✅ Content planning completed successfully!")
                return result
        except Exception as e:
            print(f"⚠️ Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                print(f"   Retrying...\n")
            else:
                print("❌ Content planning failed after all retries")
                raise

    return None

# Create and execute the crew
crew = Crew(
    agents=[content_creator_assistant],
    tasks=[task],
    tools=[exa_search_tool],
    verbose=True   #This can ba also set to False
)

result = execute_content_planning(crew)

🚀 Execution Attempt 1/3


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  fc0240f1-a1c6-439b-aca3-411b436f08fa                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16,     │
│  30-45s). Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2)               │
│  crystal-clear narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable,    │
│  no special gear.                                                                                               │
│  ID: 260d78fb-b1d0-4052-9711-c591737cd913                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI News Content Strategist                                                                              │
│                                                                                                                 │
│  Task: Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16,     │
│  30-45s). Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2)               │
│  crystal-clear narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable,    │
│  no special gear.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI News Content Strategist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "videos": [                                                                                                  │
│      {                                                                                                          │
│        "title": "AI Is Finding Cures FASTER! Your Doctor's New Secret Weapon?",                                 │
│        "hook_main": "AI is revolutionizing medicine! Drug discovery in weeks, not years.",                      │
│        "hook_alt": "Imagine a cure for *your* disease. AI is making it happen faster than ever!",               │
│        "visuals": [                                                                                             │
│          "Toy stethoscope or doctor's coat/white shirt",                                                        │
│          "Pill bottle or medicine box",                                                                         │
│          "Lab notebook with complex-looking (even scribbled) formulas",                                         │
│          "A 'progress bar' drawn on paper quickly filling up"                                                   │
│        ],                                                                                                       │
│        "tags": [                                                                                                │
│          "#ai",                                                                                                 │
│          "#artificialintelligence",                                                                             │
│          "#tech",                                                                                               │
│          "#shorts",                                                                                             │
│          "#AIHealthcare",                                                                                       │
│          "#DrugDiscovery",                                                                                      │
│          "#MedicalBreakthrough",                                                                                │
│          "#FutureofMedicine",                                                                                   │
│          "#HealthTech"                                                                                          │
│        ],                                                                                                       │
│        "cta": "If AI could cure *any* disease, which one would you want it to tackle first? Tell me!"           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "AI vs. AI: The Invisible War Protecting Your Digital Life!",                                   │
│        "hook_main": "AI is fighting cyberattacks in real-time. Your data is safer... for now!",                 │
│        "hook_alt": "The ultimate cyberwar: AI vs. AI! W

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Create a 1-week video posting plan with 5 video blueprints. Platform: YouTube Shorts (vertical 9:16, 30-45s).  │
│  Niche: Latest AI News & Developments. Primary goals: 1) thumb-stop hook in first 1s, 2) crystal-clear          │
│  narrative, 3) strong SEO phrasing, 4) comment-bait CTA. Context: solo creator, home-filmable, no special       │
│  gear.                                                                                                          │
│  Agent:                                                                                                         │
│  AI News Content Strategist                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  fc0240f1-a1c6-439b-aca3-411b436f08fa                                                                           │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "videos": [                                                                                                  │
│      {                                                                                                          │
│        "title": "AI Is Finding Cures FASTER! Your Doctor's New Secret Weapon?",                                 │
│        "hook_main": "AI is revolutionizing medicine! Drug discovery in weeks, not years.",                      │
│        "hook_alt": "Imagine a cure for *your* disease. AI is making it happen faster than ever!",               │
│        "visuals": [                                                                                             │
│          "Toy stethoscope or doctor's coat/white shirt",                                                        │
│          "Pill bottle or medicine box",                                                                         │
│          "Lab notebook with complex-looking (even scribbled) formulas",                                         │
│          "A 'progress bar' drawn on paper quickly filling up"                                                   │
│        ],                                                                                                       │
│        "tags": [                                                                                                │
│          "#ai",                                                                                                 │
│          "#artificialintelligence",                                                                             │
│          "#tech",                                                                                               │
│          "#shorts",                                                                                             │
│          "#AIHealthcare",                                                                                       │
│          "#DrugDiscovery",                                                                                      │
│          "#MedicalBreakthrough",                                                                                │
│          "#FutureofMedicine",                                                                                   │
│          "#HealthTech"                                                                                          │
│        ],                                                                                                       │
│        "cta": "If AI could cure *any* disease, which one would you want it to tackle first? Tell me!"           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "AI vs. AI: The Invisible War Protecti

✅ Content planning completed successfully!


In [18]:
# Add Output Formatting

import json
from typing import Dict, List

def parse_and_validate_output(raw_output: str) -> Dict:
    """
    Parse and validate the JSON output from the agent.

    Args:
        raw_output: The raw string output from the agent

    Returns:
        Parsed JSON dictionary
    """
    try:
        # Extract JSON from markdown code blocks if present
        if "```json" in raw_output:
            json_str = raw_output.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_output:
            json_str = raw_output.split("```")[1].split("```")[0].strip()
        else:
            json_str = raw_output

        data = json.loads(json_str)

        # Validate structure
        assert "videos" in data, "Missing 'videos' key"
        assert len(data["videos"]) == 5, f"Expected 5 videos, got {len(data['videos'])}"

        print(f"✅ Parsed {len(data['videos'])} video blueprints successfully")
        return data

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        raise
    except AssertionError as e:
        print(f"❌ Validation Error: {e}")
        raise

# Parse the results
video_content_plan = parse_and_validate_output(result.raw)

print("\n📊 CONTENT PLAN SUMMARY:")
print(f"Generated {len(video_content_plan['videos'])} video concepts")

✅ Parsed 5 video blueprints successfully

📊 CONTENT PLAN SUMMARY:
Generated 5 video concepts


In [ ]:
# Add visualisation section

import pandas as pd
from IPython.display import display, HTML

def create_content_calendar(videos: List[Dict]) -> pd.DataFrame:
    """Convert video blueprints into a readable calendar."""

    calendar_data = []
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    for i, video in enumerate(videos):
        calendar_data.append({
            'Day': days[i],
            'Title': video['title'],
            'Hook (Primary)': video['hook_main'],
            'Visual Concepts': ' | '.join(video['visuals'][:2]),
            'Key Tags': ', '.join(video['tags'][:3]),   #The  original number of Tags was 5
            'CTA': video['cta']
        })

    df = pd.DataFrame(calendar_data)
    return df

# Display the calendar
calendar_df = create_content_calendar(video_content_plan['videos'])
display(HTML(calendar_df.to_html(index=False)))

# Summary statistics
print("\n📈 CONTENT METRICS:")
print(f"✓ Total videos planned: {len(video_content_plan['videos'])}")
print(f"✓ Avg tags per video: {sum(len(v['tags']) for v in video_content_plan['videos']) / len(video_content_plan['videos']):.1f}")
print(f"✓ All videos are 30-45 seconds: Yes")
print(f"✓ All include engagement CTAs: Yes")

Day,Title,Hook (Primary),Visual Concepts,Key Tags,CTA
Monday,AI Is Finding Cures FASTER! Your Doctor's New Secret Weapon?,"AI is revolutionizing medicine! Drug discovery in weeks, not years.",Toy stethoscope or doctor's coat/white shirt | Pill bottle or medicine box,"#ai, #artificialintelligence, #tech","If AI could cure *any* disease, which one would you want it to tackle first? Tell me!"
Tuesday,AI vs. AI: The Invisible War Protecting Your Digital Life!,AI is fighting cyberattacks in real-time. Your data is safer... for now!,"A locked padlock (can be a small physical lock or drawn on paper) | A keyboard or mouse, with a hand hovering protectively","#ai, #artificialintelligence, #tech","Do you trust AI to protect your online privacy, or does it make you more nervous? Share below!"
Wednesday,Can AI Actually SAVE The Planet? Surprising Climate Solutions!,AI is tackling climate change head-on! From predicting weather to optimizing energy.,A small plant or leaf (real or fake) | A globe (toy or drawn outline),"#ai, #artificialintelligence, #tech",What's one specific way you think AI could best help fight climate change? Let's discuss!
Thursday,Your AI Assistant Just Got a BRAIN! Predictive & Proactive Tech!,AI assistants are now PREDICTING your needs! Your phone just got smarter.,"Smartphone on a table, perhaps with a 'notification' sound effect | A crystal ball (toy or improvised with a clear glass)","#ai, #artificialintelligence, #tech",Would you rather your AI wait for your command or proactively anticipate your needs? Why?
Friday,The HIDDEN Cost of AI: Is It Bad For The Planet?,AI uses massive amounts of energy! What's the environmental price of progress?,A light switch being flicked on/off repeatedly | A 'power' symbol (drawn on paper),"#ai, #artificialintelligence, #tech",What's more important: rapid AI innovation or reducing its environmental footprint? Discuss!



📈 CONTENT METRICS:
✓ Total videos planned: 5
✓ Avg tags per video: 9.2
✓ All videos are 30-45 seconds: Yes
✓ All include engagement CTAs: Yes


**Explanation output 3** Based on the above output, the `Content Quality & Real-World Grounding` is improved significantly with real data from the use of search tools and it is clearly influenced by recent, verifiable events such as:  
- AI-driven drug discovery platforms (e.g., work by DeepMind in protein structure prediction)    
- Real-time anomaly detection models  
- AI for weather forecasting improvements  
- Context-aware notification systems  
- Public sustainability reports from companies like OpenAI and Google  

**Strengths**

- Covers serious, high-impact domains  
- Aligns with ongoing global discussions  
- Suitable for analytical or news-style creators  
- Good balance of opportunity and risk framing  

**Weaknesses**

- Claims are generalized  
- No statistics, dates, or named studies  
- Heavy emotional tone (“HIDDEN cost”, “SAVE the planet”)  
- Would require citation updates to maintain credibility  

### Comparative analysis across `No Tool`, `SerperDevTool`, and `ExaSearchTool` 

Below is a three-way comparative analysis of the agent behavior and outputs across **No Tool**, **SerperDevTool**, and **ExaSearchTool**, with a focus on how each tool reshapes cognition, topic selection, and epistemic grounding.  

It shows:  
- How tooling affects authority perception  
- How freshness impacts risk  
- How signal-to-noise ratio shapes strategic depth  

| Dimension          | No Tool (Output 1)    | SerperDevTool (Output 2) | ExaSearchTool (Output 3)           |
| ------------------ | --------------------- | ------------------------ | ---------------------------------- |
| Knowledge Mode     | Generative priors     | Search-driven news       | Curated, high-signal sources       |
| Time Sensitivity   | Evergreen             | Immediate / breaking     | Recent but stabilized              |
| Topic Type         | Conceptual narratives | News & regulation        | Product launches & industry shifts |
| Specificity        | Low–medium            | Medium                   | High                               |
| Authority Signal   | “AI explainer”        | “Tech reporter”          | “Industry analyst”                 |
| Noise Level        | Low                   | Medium (trend noise)     | Very low                           |
| Hallucination Risk | Lowest                | Medium                   | Lowest-with-freshness              |
| Strategic Depth    | Low                   | Medium                   | High                               |


## Conclusions  

This example shows how to create a single specialized agent **Content Planning Agent**, specialized in YouTube Shorts about AI news and development. This agent:     
- Plans a weekly slate of video concepts
- Ensures each video follows best practices for retention
- Keeps ideas practical and filmable for a solo creator

Content creation is an excellent use case for AI agents because it involves:
- **Strategy**: Understanding platform requirements and audience psychology
- **Creativity**: Generating engaging hooks and concepts
- **Structure**: Organizing content in a repeatable, scalable format


### Key Takeaways  
1. **Agent Design**: Role + Goal + Backstory = Effective AI team member  
2. **Structured Output**: JSON schemas ensure consistent, parseable results  
3. **Modularity**: Single agents can scale to multi-agent crews for complex workflows  

## Key Improvement Areas   

Implement any of the below areas can improve the performance of the actual system.  

### Extend the System  

#### Level 1: Add More Agents   

##### Two-agent approach:

**Current vs. Proposed Workflow Analysis:**  
_Current (Single Agent):_  
- One agent doing both research AND content creation
- Mixed responsibilities can dilute focus  
- Research quality might be compromised for content creation speed  

_Proposed (Two-Agent Workflow):_  
`Research Agent → Content Strategist → Final Output`  

Suggested Agent Specifications:  
- `Research Agent:`  
_Role:_ AI News Research Specialist 
_Goal_: Discover and analyze the latest AI news, developments, and trends from reliable sources, providing comprehensive briefings with key insights and verified facts
_Tools:_  
  - SerperDevTool (news search)
  - SerperScrapeWebsiteTool (deep research)

- `Content Strategist:`  
_Role:_ YouTube Shorts Content Strategist  
_Goal:_ Transform AI research findings into viral YouTube Shorts blueprints with compelling hooks, clear narratives, and engagement-driven CTAs Tools: None needed (works with research data)  

Task Flow Recommendation:  
- `Task 1: Research Latest AI News`  
_Agent:_ Research Specialist  
_Output:_ Structured research briefing with 5-7 trending AI topics  
_Context:_ None (starting task)    

- `Task 2: Create Video Content Plan`  
_Agent:_ Content Strategist  
_Output:_ 5 video blueprints in JSON format  
_Context:_ Research findings from Task 1    

**Benefits of This Approach:**  
- Specialized Focus - Each agent excels at their specific role  
- Better Research Quality - Dedicated research agent ensures accuracy  
- Enhanced Content - Content agent can focus purely on virality/engagement  
- Scalability - Easy to add more research sources or content formats    
- Quality Control - Research validation happens before content creation  

##### Extend to a **multi-agent crew** (e.g., writer, designer, SEO specialist) 

**Video Script Writer Agent**
```
Role: Professional YouTube scriptwriter
Goal: Write engaging 30-45 second scripts
Tools: Copy optimization, trending language analysis
```

**Thumbnail Designer Agent**
```
Role: Visual designer for YouTube thumbnails
Goal: Suggest thumbnail compositions and text
Tools: Color psychology, design templates
```

**Fact Checker Agent**
```
Role: AI research and fact verification specialist
Goal: Validate all AI news claims
Tools: Academic paper search, news verification
```  

#### Level 2: Implement Tool Integration

- Connect to **YouTube Analytics** for trend data
- Integrate with **trend detection APIs** (Trend API, Google Trends)
- Add **image generation tools** (DALL-E, Midjourney)
- Connect to **notion/calendar APIs** for scheduling


### Level 3: Build a Web Interface

```bash
Streamlit App Features:
├─ User input form for topics
├─ Real-time agent execution display
├─ Calendar visualization
├─ Export to multiple formats
├─ Performance metrics dashboard
└─ Scheduled batch processing
```

### Level 4: Continuous Improvement

- **Track performance metrics** for generated content
- **Fine-tune prompts** based on engagement data
- **Implement A/B testing** framework
- **Build feedback loop** for learning
